# Notebook 00 — Dataset Exploration

**NeuroDriver CNN ADAS Colombia** — academic research prototype.

Status: dataset preparation / audit stage (Phase 1). **No CNN training happens in this notebook.**


## 1. NeuroDriver context

This project is a scoped-down academic derivative of the broader NeuroDriver ADAS research effort.
NeuroDriver as a whole aims at a full advanced driver-assistance stack; this prototype isolates a
single sub-problem: **frame-level visual perception via a compact CNN**, useful as a building block
for future perception/alert features. It does not implement steering, throttle, braking, or any
CAN-level vehicle control.


## 2. Colombian problem focus

The end goal is a perception model relevant to **Colombian urban/road driving conditions**
(vehicle mix, informal traffic patterns, motorcycle density, road/infrastructure differences).
Colombian dashcam data specific to NeuroDriver is not yet integrated into this repository — see
`docs/colombian_domain_strategy.md` for the planned integration path.


## 3. Scope reduction

Out of scope for this prototype: object-detector training (YOLO/SSD/Faster R-CNN/RetinaNet/Detectron),
FastAPI/frontend implementation, mobile/Raspberry Pi deployment, TFLite/ONNX export, CAN control, and
long production training runs. In scope: frame-level 4-class classification, BDD100K as a source domain,
MobileNetV2 transfer learning, and Knowledge-Distillation-ready scaffolding.


## 4. BDD100K as provisional SOURCE domain

**BDD100K is not Colombian data.** It is used here only as an initial, large, diverse SOURCE DOMAIN
to bootstrap the CNN before Colombian NeuroDriver TARGET DOMAIN data is available. Any distribution or
performance number in this notebook describes BDD100K only.


## 5. Colombia as future TARGET domain

The planned research progression is:

```
ImageNet MobileNetV2 -> BDD100K source-domain learning -> NeuroDriver Colombian dashcam data
-> Colombian fine-tuning -> Knowledge Distillation (NeuroDriver Teacher -> MobileNetV2 Student)
-> compact Student -> web perception/alert service
```

See `docs/colombian_domain_strategy.md` for the full 9-step plan.


## 6. Four-class task

Exact classification target (frame-level, not object detection):

| Class | Index | Definition |
|---|---|---|
| CLEAR | 0 | no relevant vehicle and no relevant pedestrian |
| VEHICLE | 1 | >=1 relevant vehicle, no relevant pedestrian |
| PEDESTRIAN | 2 | >=1 relevant pedestrian, no relevant vehicle |
| MIXED | 3 | >=1 relevant vehicle and >=1 relevant pedestrian |

Initial category mapping: vehicle = {car, truck, bus, motorcycle}; pedestrian = {pedestrian};
auxiliary (excluded from vehicle/pedestrian) = {rider, bicycle}.


## 7. Motorcycle handling policy

Motorcycles count toward `VEHICLE` in the 4-class output, but are tracked explicitly via
`has_motorcycle` / `num_motorcycles` fields in the manifest so a **motorcycle-containing-frame
evaluation slice** remains possible later (`evaluate_motorcycle_subset`, see
`src/neurodriver_cnn/evaluation/metrics.py`). Motorcycle representation is never silently merged
away, and `rider`/`bicycle` are never silently remapped into vehicle, pedestrian, or motorcycle.


## 8. Full Frame labeling method

Every mapped object anywhere in the full image contributes to the frame label. Implemented in
`neurodriver_cnn.labeling.frame_labels.classify_fullframe`.


## 9. ADAS ROI labeling method

Only objects relevant to a configurable forward-driving-corridor Region Of Interest contribute.
A box is ROI-relevant if its center falls inside the ROI, or if
`intersection_area / bbox_area >= threshold`. Initial configuration (see
`configs/dataset_config.json`):

```json
{
  "x_min": 0.20, "x_max": 0.80,
  "y_min": 0.35, "y_max": 1.00,
  "bbox_intersection_threshold": 0.35
}
```

Implemented in `neurodriver_cnn.labeling.frame_labels.classify_roi` /
`neurodriver_cnn.labeling.roi`.


In [ ]:
import sys
from pathlib import Path

# Resolve project root without hardcoding a personal path (works locally and in Colab
# once the repo is cloned/mounted).
PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "configs").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import json
import pandas as pd

from neurodriver_cnn.config import load_dataset_config

dataset_config = load_dataset_config(PROJECT_ROOT)
print(json.dumps(dataset_config, indent=2))


## 10. Dataset audit

Loads `reports/dataset_audit.json` (produced by `scripts/03_analyze_manifest.py`).

In [ ]:
audit_path = PROJECT_ROOT / "reports" / "dataset_audit.json"
if audit_path.exists():
    audit = json.loads(audit_path.read_text(encoding="utf-8"))
    print(json.dumps(audit, indent=2)[:3000])
else:
    audit = None
    print("PENDING: reports/dataset_audit.json not found yet.")
    print("Run scripts/00-03 after placing BDD100K data (see docs/bdd100k_setup.md).")


## 11. Full Frame vs ROI distribution comparison

Real figures are generated by `scripts/03_analyze_manifest.py` once BDD100K data is available.

In [ ]:
from IPython.display import Image, display

for fig_name in ["class_distribution_fullframe.png", "class_distribution_roi.png", "fullframe_vs_roi.png"]:
    fig_path = PROJECT_ROOT / "reports" / "figures" / fig_name
    if fig_path.exists():
        display(Image(filename=str(fig_path)))
    else:
        print(f"PENDING: {fig_name} not generated yet.")


## 12. Visual examples

`class_examples.png` (per-class frames) and `roi_examples.png` (ROI overlay) from the same audit script.

In [ ]:
for fig_name in ["class_examples.png", "roi_examples.png", "motorcycle_distribution.png"]:
    fig_path = PROJECT_ROOT / "reports" / "figures" / fig_name
    if fig_path.exists():
        display(Image(filename=str(fig_path)))
    else:
        print(f"PENDING: {fig_name} not generated yet.")


## 13. Splits and leakage prevention

Rules enforced by `neurodriver_cnn.data.manifest` and `scripts/05_validate_dataset.py`:

- split assigned **before** any augmentation;
- augmentation applied to TRAIN only (see Notebook 01);
- TEST is never used for tuning;
- splitting is **group-aware** (`group_aware_split`) using real BDD100K `videoName` when present,
  so frames from the same video/run never cross TRAIN/VALIDATION;
- official BDD TRAIN -> internal TRAIN + VALIDATION; official BDD VAL -> academic TEST;
- seed = 42 everywhere.


## 14. Domain-shift limitations (BDD100K -> Colombia)

Hypotheses to validate once Colombian NeuroDriver data is available (never assumed true here):

- different vehicle mix (higher motorcycle density expected in Colombia);
- different road infrastructure/signage;
- different pedestrian behavior/context;
- different weather/lighting distribution;
- different camera mounting/field of view.

See `docs/colombian_domain_strategy.md`.


## 15. Experimental subset

Loads `reports/sampling_plan.md` (produced by `scripts/04_build_experiment_subset.py`).

In [ ]:
sampling_plan_path = PROJECT_ROOT / "reports" / "sampling_plan.md"
if sampling_plan_path.exists():
    print(sampling_plan_path.read_text(encoding="utf-8"))
else:
    print("PENDING: reports/sampling_plan.md not found yet.")


## 16. Conclusions

- The dataset pipeline (parser, ROI/frame labeling, Common Manifest, audit, subset, validation) is
  implemented and unit-tested independently of whether BDD100K data is physically present.
- **BDD100K availability in this environment:** see the environment-check output from
  `scripts/00_check_environment.py` and the PENDING markers above.
- Full Frame vs ROI has **not** been permanently chosen; that decision is recorded in
  `docs/decisions_log.md` once real distributions/examples exist.
- Next notebook (`01_cnn_baseline_mobilenetv2.ipynb`) builds the simple CNN baseline and the
  KD-ready MobileNetV2 Student architecture — still no full training in this milestone.
